# Topology Kaggle Seed44

Recovered original Kaggle workflow. Run setup first; restart after package installation if requested, then run the training and export cells in order. Requires the original Kaggle input package and CUDA GPU. Outputs have been cleared; no training was rerun during extraction.


## Source cell 103

Original notebook zero-based index: 102.


In [ ]:
# CELL 1 — Match the five recorded package versions.
# Enable Internet in Kaggle settings before running this cell.
from pathlib import Path
import json
import sys
import subprocess
import importlib.metadata as md

BASE = Path('/kaggle/input/datasets/amiramohammeddd/ddi-unique263-cal-test-stage2/DDI_KAGGLE_TRANSFER_SEED43/DDI_V2')
reference_path = BASE / "PRIMARY_TOPOLOGY_TRAIN_ONLY_V2/training_current_node_bn_v2/original/seed_42/config.json"
assert reference_path.is_file(), f"Missing reference: {reference_path}"
reference = json.loads(reference_path.read_text())
expected = reference["versions"]
def resolve_graph(filename, expected_hash=None):
    import hashlib
    import zipfile
    import os

    input_root = Path("/kaggle/input/datasets/amiramohammeddd/ddi-unique263-cal-test-stage2")
    unpacked = Path("/kaggle/working/SEED44_EXTRA_GRAPHS")
    unpacked.mkdir(parents=True, exist_ok=True)

    def sha(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for block in iter(lambda: f.read(1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    # Kaggle may keep the upload zipped or expose already-extracted files.
    # Read only matching graph members; never extract arbitrary archive paths.
    for archive_path in sorted(input_root.rglob("SEED44_EXTRA_GRAPHS*.zip")):
        with zipfile.ZipFile(archive_path, "r") as archive:
            for member in archive.infolist():
                if member.is_dir() or Path(member.filename).name != filename:
                    continue
                contents = archive.read(member)
                fingerprint = hashlib.sha256(contents).hexdigest()
                destination = unpacked / (fingerprint + "_" + filename)
                if destination.exists():
                    assert sha(destination) == fingerprint, "Extracted graph is damaged."
                else:
                    temporary = destination.with_suffix(".npz.tmp")
                    temporary.write_bytes(contents)
                    os.replace(temporary, destination)

    candidates = sorted(set(
        [p for p in Path("/kaggle/input").rglob(filename) if p.is_file()]
        + [p for p in unpacked.glob("*_" + filename) if p.is_file()]
    ))
    assert candidates, f"Missing graph: {filename}; attach SEED44_EXTRA_GRAPHS.zip."
    hashes = {p: sha(p) for p in candidates}
    if expected_hash is not None:
        candidates = [p for p in candidates if hashes[p] == expected_hash]
        assert candidates, f"No graph matches the recorded hash: {filename}"
    assert len({hashes[p] for p in candidates}) == 1, (
        f"Several different graphs named {filename}; report their paths: {candidates}"
    )
    return candidates[0]
for name in ["original.npz", "rewired_seed_44.npz", "self_loops_only.npz"]:
    expected_hash = reference["graph_sha256"] if name == "original.npz" else None
    print("Graph found:", resolve_graph(name, expected_hash))
packages = ["torch", "torch-geometric", "numpy", "pandas", "scikit-learn"]

def installed(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None

changes = {p: [installed(p), expected[p]] for p in packages if installed(p) != expected[p]}
print("Required changes:", changes)
if changes:
    if "torch" in changes:
        assert expected["torch"].endswith("+cu128"), "Unexpected Torch build; stop and report."
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "--upgrade",
            "torch==" + expected["torch"],
            "--index-url", "https://download.pytorch.org/whl/cu128"
        ])
    pins = [p + "==" + expected[p] for p in packages if p != "torch"]
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade", *pins
    ])
    for package in packages:
        assert installed(package) == expected[package], (
            f"Version mismatch: {package}: {installed(package)} != {expected[package]}"
        )
    raise RuntimeError(
        "INSTALLATION COMPLETE. Restart the Kaggle kernel/session once, "
        "then run this notebook again from cell 1. Do not delete the session."
    )

import torch
assert torch.cuda.is_available(), "Enable GPU in Kaggle settings."
print("Environment matched ")
print("GPU:", torch.cuda.get_device_name(0))
print("Next: train Original 44, Rewired 44, then Self-loops only 44.")


## Source cell 104

Original notebook zero-based index: 103.


In [ ]:
from pathlib import Path
import os
import json
import hashlib
import time
import math
import gc
import importlib.metadata as md
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
BASE = Path('/kaggle/input/datasets/amiramohammeddd/ddi-unique263-cal-test-stage2/DDI_KAGGLE_TRANSFER_SEED43/DDI_V2')
FEATURES = BASE / "PRIMARY_TOPOLOGY_TRAIN_ONLY_V2"
REFERENCE = FEATURES / "training_current_node_bn_v2/original/seed_42"
RUN = Path("/kaggle/working/DDI_V2/PRIMARY_TOPOLOGY_TRAIN_ONLY_V2/training_current_node_bn_v2/original/seed_44")
def resolve_graph(filename, expected_hash=None):
    import hashlib
    import zipfile
    import os

    input_root = Path("/kaggle/input/datasets/amiramohammeddd/ddi-unique263-cal-test-stage2")
    unpacked = Path("/kaggle/working/SEED44_EXTRA_GRAPHS")
    unpacked.mkdir(parents=True, exist_ok=True)

    def sha(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for block in iter(lambda: f.read(1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    # Kaggle may keep the upload zipped or expose already-extracted files.
    # Read only matching graph members; never extract arbitrary archive paths.
    for archive_path in sorted(input_root.rglob("SEED44_EXTRA_GRAPHS*.zip")):
        with zipfile.ZipFile(archive_path, "r") as archive:
            for member in archive.infolist():
                if member.is_dir() or Path(member.filename).name != filename:
                    continue
                contents = archive.read(member)
                fingerprint = hashlib.sha256(contents).hexdigest()
                destination = unpacked / (fingerprint + "_" + filename)
                if destination.exists():
                    assert sha(destination) == fingerprint, "Extracted graph is damaged."
                else:
                    temporary = destination.with_suffix(".npz.tmp")
                    temporary.write_bytes(contents)
                    os.replace(temporary, destination)

    candidates = sorted(set(
        [p for p in Path("/kaggle/input").rglob(filename) if p.is_file()]
        + [p for p in unpacked.glob("*_" + filename) if p.is_file()]
    ))
    assert candidates, f"Missing graph: {filename}; attach SEED44_EXTRA_GRAPHS.zip."
    hashes = {p: sha(p) for p in candidates}
    if expected_hash is not None:
        candidates = [p for p in candidates if hashes[p] == expected_hash]
        assert candidates, f"No graph matches the recorded hash: {filename}"
    assert len({hashes[p] for p in candidates}) == 1, (
        f"Several different graphs named {filename}; report their paths: {candidates}"
    )
    return candidates[0]

GRAPH = resolve_graph("original.npz", json.loads((REFERENCE / "config.json").read_text())["graph_sha256"])
SEED = 44
EPOCHS = 120
BATCH_SIZE = 512
SAVE_EVERY = 50
assert torch.cuda.is_available(), "A CUDA GPU is required."
assert BASE.exists(), "Input folder missing."
DEVICE = torch.device("cuda")
RUN.mkdir(parents=True, exist_ok=True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
def save_json(path, obj):
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False))
    os.replace(tmp, path)
def save_pt(path, obj):
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)
def save_history(history):
    tmp = RUN / "training_history.csv.tmp"
    pd.DataFrame(history).to_csv(tmp, index=False)
    os.replace(tmp, RUN / "training_history.csv")
class CurrentNodeBNGCN(nn.Module):
    def __init__(self):
        super().__init__()
        dims = [2250, 384, 384, 192]
        self.convs = nn.ModuleList([
            GCNConv(
                dims[i],
                dims[i + 1],
                improved=False,
                cached=False,
                add_self_loops=False,
                normalize=True,
                bias=True
            )
            for i in range(3)
        ])
        self.norms = nn.ModuleList([
            nn.BatchNorm1d(d, track_running_stats=False)
            for d in [384, 384, 192]
        ])
        self.head = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(256, 84)
        )
    def encode(self, x, edges):
        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            x = norm(conv(x, edges))
            if i < 2:
                x = F.dropout(
                    F.relu(x),
                    p=0.35,
                    training=self.training
                )
        return x
    def classify(self, z, pairs):
        a = z[pairs[:, 0]]
        b = z[pairs[:, 1]]
        pair_features = torch.cat(
            [a, b, (a - b).abs(), a * b],
            dim=1
        )
        return self.head(pair_features)
    def forward(self, x, edges, pairs):
        return self.classify(self.encode(x, edges), pairs)
def main():
    reference_path = REFERENCE / "config.json"
    cache = FEATURES / "train_only_feature_cache.npz"
    for path in [reference_path, cache, GRAPH]:
        assert path.exists(), f"Missing file: {path}"
    reference = json.loads(reference_path.read_text())
    assert reference["condition"] == "original"
    assert reference["seed"] == 42
    assert digest(cache) == reference["feature_sha256"]
    assert digest(GRAPH) == reference["graph_sha256"]
    versions = {
        package: md.version(package)
        for package in [
            "torch",
            "torch-geometric",
            "numpy",
            "pandas",
            "scikit-learn"
        ]
    }
    original_v2_path = (
        FEATURES / "training_current_node_bn_v2/original/seed_42/config.json"
    )
    assert original_v2_path.exists(), "Corrected Original 42 config missing."
    original_v2 = json.loads(original_v2_path.read_text())
    assert original_v2["protocol"] == "current_node_bn_v2"
    assert original_v2["condition"] == "original"
    assert original_v2["seed"] == 42
    assert original_v2["feature_sha256"] == reference["feature_sha256"]
    assert original_v2["split_sha256"] == reference["split_sha256"]
    differences = {
        package: [original_v2["versions"].get(package), version]
        for package, version in versions.items()
        if original_v2["versions"].get(package) != version
    }
    assert not differences, (
        f"Environment differs from corrected Original 42: {differences}"
    )
    print("Environment matches corrected Original 42 ")
    with np.load(cache, allow_pickle=False) as loaded:
        x = torch.tensor(
            loaded["x"],
            dtype=torch.float32,
            device=DEVICE
        )
        drugs = loaded["all_drugs"].tolist()
    assert x.shape == (1705, 2250), f"Unexpected shape: {x.shape}"
    assert torch.isfinite(x).all()
    mapping = {drug: i for i, drug in enumerate(drugs)}
    assert len(mapping) == len(drugs), "Duplicate drug IDs in cache."
    data = {}
    # Only Train and Calibration are read. Test is not read.
    for split in ["train", "calibration"]:
        path = BASE / f"clean_{split}_split.csv"
        assert path.exists(), f"Missing file: {path}"
        assert digest(path) == reference["split_sha256"][split]
        frame = pd.read_csv(path)
        a = frame["Drug1"].map(mapping)
        b = frame["Drug2"].map(mapping)
        assert a.notna().all() and b.notna().all(), (
            f"Unmapped drug IDs in {split}."
        )
        labels = frame["label_id"].to_numpy(dtype=np.int64)
        assert set(labels) == set(range(84)), (
            f"Unexpected class coverage in {split}."
        )
        data[split] = {
            "pairs": torch.tensor(
                np.column_stack([a, b]).astype(np.int64),
                dtype=torch.long,
                device=DEVICE
            ),
            "y": torch.tensor(
                labels,
                dtype=torch.long,
                device=DEVICE
            ),
            "truth": labels
        }
    with np.load(GRAPH, allow_pickle=False) as loaded:
        edges = torch.tensor(
            loaded["edge_index"],
            dtype=torch.long,
            device=DEVICE
        )
    assert edges.ndim == 2 and edges.shape[0] == 2
    assert edges.numel() > 0
    assert int(edges.min()) >= 0
    assert int(edges.max()) < len(drugs)
    counts = np.bincount(
        data["train"]["truth"],
        minlength=84
    ).astype(float)
    weights = np.clip(
        np.sqrt(counts.max() / (counts + 1)),
        0.5,
        3.0
    )
    weights /= weights.mean()
    np.testing.assert_allclose(
        weights,
        reference["class_weights"],
        rtol=0,
        atol=1e-12
    )
    loss_fn = nn.CrossEntropyLoss(
        weight=torch.tensor(
            weights,
            dtype=torch.float32,
            device=DEVICE
        )
    )
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    # Fresh initialization; no Original or Rewired weights loaded.
    model = CurrentNodeBNGCN().to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.0007,
        weight_decay=0.0001
    )
    assert sum(p.numel() for p in model.parameters()) == 1636244
    assert all(
        not module.track_running_stats
        for module in model.modules()
        if isinstance(module, nn.BatchNorm1d)
    )
    config = {
        "protocol": "current_node_bn_v2",
        "condition": "original",
        "seed": SEED,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "optimizer": "AdamW",
        "learning_rate": 0.0007,
        "weight_decay": 0.0001,
        "gradient_clip_norm": 5.0,
        "early_stopping": None,
        "scheduler": None,
        "architecture": str(model),
        "class_weights": weights.tolist(),
        "primary_selection": "calibration_macro_f1_all84",
        "tie_rule": "earliest epoch",
        "BN": "current full-node statistics in train and eval",
        "BN_track_running_stats": False,
        "BN_node_scope": "all 1705 fixed nodes; transductive",
        "descriptor_preprocessing_fit": "Train nodes only",
        "graph_source": "Train edges only",
        "shuffle_seed": "seed + 100000 * epoch",
        "feature_sha256": digest(cache),
        "graph_sha256": digest(GRAPH),
        "split_sha256": reference["split_sha256"],
        "versions": versions,
        "gpu": torch.cuda.get_device_name(0),
        "test_evaluated": False,
        "bitwise_GPU_resume_guaranteed": False
    }
    # All model and training settings must match corrected Original 42.
    for key in [
        "protocol", "epochs", "batch_size", "optimizer",
        "learning_rate", "weight_decay", "gradient_clip_norm",
        "early_stopping", "scheduler", "architecture", "class_weights",
        "primary_selection", "tie_rule", "BN", "BN_track_running_stats",
        "BN_node_scope", "descriptor_preprocessing_fit", "shuffle_seed"
    ]:
        assert config[key] == original_v2[key], f"Original mismatch: {key}"
    config_path = RUN / "config.json"
    if config_path.exists():
        saved_config = json.loads(config_path.read_text())
        differences = {
            key: [saved_config.get(key), config.get(key)]
            for key in set(saved_config) | set(config)
            if saved_config.get(key) != config.get(key)
        }
        assert set(differences) <= {"gpu"}, (
            f"Training configuration changed: {differences}"
        )
        # Preserve the exact original config bytes and checkpoint hash.
        config = saved_config
    else:
        save_json(config_path, config)
    # Append hardware provenance separately; never edit the original config.
    import platform
    from datetime import datetime, timezone
    provenance_path = RUN / "execution_segments.json"
    segments = json.loads(provenance_path.read_text()) if provenance_path.exists() else []
    segments.append({
        "utc": datetime.now(timezone.utc).isoformat(),
        "platform": "Kaggle",
        "python": platform.python_version(),
        "gpu": torch.cuda.get_device_name(0),
        "torch_cuda": torch.version.cuda,
        "versions": versions,
        "original_config_gpu": config["gpu"],
        "bitwise_continuation_guaranteed": False,
        "config_sha256": digest(config_path)
    })
    save_json(provenance_path, segments)
    config_hash = digest(config_path)
    done = RUN / "TRAINING_COMPLETE.json"
    if done.exists():
        completed = json.loads(done.read_text())
        assert completed["config_sha256"] == config_hash
        print("Run already completed.")
        print(json.dumps(completed, indent=2))
        print("\nSaved:", RUN)
        return
    last = RUN / "last_training_state.pt"
    epoch = 1
    next_batch = 0
    loss_sum = 0.0
    loss_batches = 0
    history = []
    best = {
        metric: -1.0
        for metric in ["macro_f1", "accuracy", "weighted_f1"]
    }
    best_epochs = {metric: None for metric in best}
    elapsed_before = 0.0
    if last.exists():
        saved = torch.load(
            last,
            map_location="cpu",
            weights_only=True
        )
        assert saved["config_sha256"] == config_hash
        model.load_state_dict(saved["model"])
        optimizer.load_state_dict(saved["optimizer"])
        epoch = saved["epoch"]
        next_batch = saved["next_batch"]
        loss_sum = saved["loss_sum"]
        loss_batches = saved["loss_batches"]
        history = saved["history"]
        best = saved["best"]
        best_epochs = saved["best_epochs"]
        elapsed_before = saved["elapsed_seconds"]
        torch.set_rng_state(saved["cpu_rng"])
        torch.cuda.set_rng_state(saved["cuda_rng"])
        del saved
        print(
            f"RESUME: Original 44 — epoch {epoch}, "
            f"completed batches {next_batch}"
        )
    else:
        print("START: Original 44 — corrected BN — FROM SCRATCH")
    started = time.perf_counter()
    def checkpoint():
        save_pt(last, {
            "config_sha256": config_hash,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "next_batch": next_batch,
            "loss_sum": loss_sum,
            "loss_batches": loss_batches,
            "history": history,
            "best": best,
            "best_epochs": best_epochs,
            "cpu_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state(),
            "elapsed_seconds": (
                elapsed_before + time.perf_counter() - started
            )
        })
    @torch.no_grad()
    def calibration_metrics():
        # Dropout off; BN uses current full-node statistics.
        model.eval()
        z = model.encode(x, edges)
        assert torch.isfinite(z).all()
        predictions = []
        pairs = data["calibration"]["pairs"]
        for start in range(0, len(pairs), 2048):
            logits = model.classify(z, pairs[start:start + 2048])
            assert torch.isfinite(logits).all()
            predictions.append(logits.argmax(1).cpu().numpy())
        pred = np.concatenate(predictions)
        truth = data["calibration"]["truth"]
        return {
            "accuracy": float(accuracy_score(truth, pred)),
            "macro_f1": float(f1_score(
                truth,
                pred,
                labels=np.arange(84),
                average="macro",
                zero_division=0
            )),
            "weighted_f1": float(f1_score(
                truth,
                pred,
                labels=np.arange(84),
                average="weighted",
                zero_division=0
            ))
        }
    n = len(data["train"]["pairs"])
    n_batches = math.ceil(n / BATCH_SIZE)
    progress = tqdm(
        total=EPOCHS,
        initial=min(epoch - 1, EPOCHS),
        desc="Original 44 — current-node BN"
    )
    while epoch <= EPOCHS:
        epoch_start = time.perf_counter()
        resumed = next_batch > 0
        generator = torch.Generator().manual_seed(
            SEED + 100000 * epoch
        )
        order = torch.randperm(
            n,
            generator=generator
        ).to(DEVICE)
        model.train()
        batches = tqdm(
            total=n_batches,
            initial=next_batch,
            desc=f"Epoch {epoch}/{EPOCHS}",
            leave=False
        )
        for batch in range(next_batch, n_batches):
            idx = order[
                batch * BATCH_SIZE:(batch + 1) * BATCH_SIZE
            ]
            optimizer.zero_grad(set_to_none=True)
            logits = model(
                x,
                edges,
                data["train"]["pairs"][idx]
            )
            loss = loss_fn(logits, data["train"]["y"][idx])
            assert torch.isfinite(loss), "Nonfinite loss."
            loss.backward()
            grad_norm = nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )
            assert torch.isfinite(grad_norm), "Nonfinite gradients."
            optimizer.step()
            loss_sum += float(loss.detach())
            loss_batches += 1
            next_batch = batch + 1
            batches.update(1)
            if next_batch % 10 == 0:
                batches.set_postfix(
                    loss=f"{loss_sum / loss_batches:.4f}"
                )
            if (
                next_batch % SAVE_EVERY == 0
                or next_batch == n_batches
            ):
                checkpoint()
        batches.close()
        metrics = calibration_metrics()
        for metric, value in metrics.items():
            if value > best[metric]:
                best[metric] = value
                best_epochs[metric] = epoch
                save_pt(RUN / f"best_by_{metric}.pt", {
                    "config_sha256": config_hash,
                    "model": model.state_dict(),
                    "epoch": epoch,
                    "criterion": metric,
                    "calibration_metrics": metrics,
                    "test_evaluated": False
                })
        row = {
            "epoch": epoch,
            "mean_batch_train_loss": loss_sum / loss_batches,
            "cal_accuracy": metrics["accuracy"],
            "cal_macro_f1": metrics["macro_f1"],
            "cal_weighted_f1": metrics["weighted_f1"],
            "epoch_segment_seconds": (
                time.perf_counter() - epoch_start
            ),
            "resumed_partway": resumed
        }
        history.append(row)
        epoch += 1
        next_batch = 0
        loss_sum = 0.0
        loss_batches = 0
        checkpoint()
        save_history(history)
        progress.update(1)
        progress.set_postfix(
            best_macro=f"{100 * best['macro_f1']:.2f}%"
        )
        tqdm.write(
            f"Epoch {row['epoch']:03d}/{EPOCHS} | "
            f"loss={row['mean_batch_train_loss']:.4f} | "
            f"Cal Acc={100 * metrics['accuracy']:.2f}% | "
            f"Macro-F1={100 * metrics['macro_f1']:.2f}% | "
            f"Weighted-F1={100 * metrics['weighted_f1']:.2f}% | saved"
        )
    progress.close()
    save_history(history)
    result = {
        "protocol": "current_node_bn_v2",
        "condition": "original",
        "seed": SEED,
        "completed_epochs": EPOCHS,
        "best_calibration_scores": best,
        "best_epochs": best_epochs,
        "primary_checkpoint": "best_by_macro_f1.pt",
        "elapsed_hours": (
            elapsed_before + time.perf_counter() - started
        ) / 3600,
        "test_evaluated": False,
        "config_sha256": config_hash
    }
    save_json(done, result)
    print("\n=== TRAINING COMPLETE ===")
    print(json.dumps(result, indent=2))
    print("\nSaved:", RUN)
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
main()


## Source cell 105

Original notebook zero-based index: 104.


In [ ]:
from pathlib import Path
import os
import json
import hashlib
import time
import math
import gc
import importlib.metadata as md
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
BASE = Path('/kaggle/input/datasets/amiramohammeddd/ddi-unique263-cal-test-stage2/DDI_KAGGLE_TRANSFER_SEED43/DDI_V2')
FEATURES = BASE / "PRIMARY_TOPOLOGY_TRAIN_ONLY_V2"
REFERENCE = FEATURES / "training_current_node_bn_v2/original/seed_42"
RUN = Path("/kaggle/working/DDI_V2/PRIMARY_TOPOLOGY_TRAIN_ONLY_V2/training_current_node_bn_v2/rewired/seed_44")
def resolve_graph(filename, expected_hash=None):
    import hashlib
    import zipfile
    import os

    input_root = Path("/kaggle/input/datasets/amiramohammeddd/ddi-unique263-cal-test-stage2")
    unpacked = Path("/kaggle/working/SEED44_EXTRA_GRAPHS")
    unpacked.mkdir(parents=True, exist_ok=True)

    def sha(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for block in iter(lambda: f.read(1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    # Kaggle may keep the upload zipped or expose already-extracted files.
    # Read only matching graph members; never extract arbitrary archive paths.
    for archive_path in sorted(input_root.rglob("SEED44_EXTRA_GRAPHS*.zip")):
        with zipfile.ZipFile(archive_path, "r") as archive:
            for member in archive.infolist():
                if member.is_dir() or Path(member.filename).name != filename:
                    continue
                contents = archive.read(member)
                fingerprint = hashlib.sha256(contents).hexdigest()
                destination = unpacked / (fingerprint + "_" + filename)
                if destination.exists():
                    assert sha(destination) == fingerprint, "Extracted graph is damaged."
                else:
                    temporary = destination.with_suffix(".npz.tmp")
                    temporary.write_bytes(contents)
                    os.replace(temporary, destination)

    candidates = sorted(set(
        [p for p in Path("/kaggle/input").rglob(filename) if p.is_file()]
        + [p for p in unpacked.glob("*_" + filename) if p.is_file()]
    ))
    assert candidates, f"Missing graph: {filename}; attach SEED44_EXTRA_GRAPHS.zip."
    hashes = {p: sha(p) for p in candidates}
    if expected_hash is not None:
        candidates = [p for p in candidates if hashes[p] == expected_hash]
        assert candidates, f"No graph matches the recorded hash: {filename}"
    assert len({hashes[p] for p in candidates}) == 1, (
        f"Several different graphs named {filename}; report their paths: {candidates}"
    )
    return candidates[0]

GRAPH = resolve_graph("rewired_seed_44.npz")
SEED = 44
EPOCHS = 120
BATCH_SIZE = 512
SAVE_EVERY = 50
assert torch.cuda.is_available(), "A CUDA GPU is required."
assert BASE.exists(), "Input folder missing."
DEVICE = torch.device("cuda")
RUN.mkdir(parents=True, exist_ok=True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
def save_json(path, obj):
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False))
    os.replace(tmp, path)
def save_pt(path, obj):
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)
def save_history(history):
    tmp = RUN / "training_history.csv.tmp"
    pd.DataFrame(history).to_csv(tmp, index=False)
    os.replace(tmp, RUN / "training_history.csv")
class CurrentNodeBNGCN(nn.Module):
    def __init__(self):
        super().__init__()
        dims = [2250, 384, 384, 192]
        self.convs = nn.ModuleList([
            GCNConv(
                dims[i],
                dims[i + 1],
                improved=False,
                cached=False,
                add_self_loops=False,
                normalize=True,
                bias=True
            )
            for i in range(3)
        ])
        self.norms = nn.ModuleList([
            nn.BatchNorm1d(d, track_running_stats=False)
            for d in [384, 384, 192]
        ])
        self.head = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(256, 84)
        )
    def encode(self, x, edges):
        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            x = norm(conv(x, edges))
            if i < 2:
                x = F.dropout(
                    F.relu(x),
                    p=0.35,
                    training=self.training
                )
        return x
    def classify(self, z, pairs):
        a = z[pairs[:, 0]]
        b = z[pairs[:, 1]]
        pair_features = torch.cat(
            [a, b, (a - b).abs(), a * b],
            dim=1
        )
        return self.head(pair_features)
    def forward(self, x, edges, pairs):
        return self.classify(self.encode(x, edges), pairs)
def main():
    reference_path = REFERENCE / "config.json"
    cache = FEATURES / "train_only_feature_cache.npz"
    for path in [reference_path, cache, GRAPH]:
        assert path.exists(), f"Missing file: {path}"
    reference = json.loads(reference_path.read_text())
    assert reference["condition"] == "original"
    assert reference["seed"] == 42
    assert digest(cache) == reference["feature_sha256"]
    versions = {
        package: md.version(package)
        for package in [
            "torch",
            "torch-geometric",
            "numpy",
            "pandas",
            "scikit-learn"
        ]
    }
    original_v2_path = (
        FEATURES / "training_current_node_bn_v2/original/seed_42/config.json"
    )
    assert original_v2_path.exists(), "Corrected Original 42 config missing."
    original_v2 = json.loads(original_v2_path.read_text())
    assert original_v2["protocol"] == "current_node_bn_v2"
    assert original_v2["condition"] == "original"
    assert original_v2["seed"] == 42
    assert original_v2["feature_sha256"] == reference["feature_sha256"]
    assert original_v2["split_sha256"] == reference["split_sha256"]
    differences = {
        package: [original_v2["versions"].get(package), version]
        for package, version in versions.items()
        if original_v2["versions"].get(package) != version
    }
    assert not differences, (
        f"Environment differs from corrected Original 42: {differences}"
    )
    print("Environment matches corrected Original 42 ")
    with np.load(cache, allow_pickle=False) as loaded:
        x = torch.tensor(
            loaded["x"],
            dtype=torch.float32,
            device=DEVICE
        )
        drugs = loaded["all_drugs"].tolist()
    assert x.shape == (1705, 2250), f"Unexpected shape: {x.shape}"
    assert torch.isfinite(x).all()
    mapping = {drug: i for i, drug in enumerate(drugs)}
    assert len(mapping) == len(drugs), "Duplicate drug IDs in cache."
    data = {}
    # Only Train and Calibration are read. Test is not read.
    for split in ["train", "calibration"]:
        path = BASE / f"clean_{split}_split.csv"
        assert path.exists(), f"Missing file: {path}"
        assert digest(path) == reference["split_sha256"][split]
        frame = pd.read_csv(path)
        a = frame["Drug1"].map(mapping)
        b = frame["Drug2"].map(mapping)
        assert a.notna().all() and b.notna().all(), (
            f"Unmapped drug IDs in {split}."
        )
        labels = frame["label_id"].to_numpy(dtype=np.int64)
        assert set(labels) == set(range(84)), (
            f"Unexpected class coverage in {split}."
        )
        data[split] = {
            "pairs": torch.tensor(
                np.column_stack([a, b]).astype(np.int64),
                dtype=torch.long,
                device=DEVICE
            ),
            "y": torch.tensor(
                labels,
                dtype=torch.long,
                device=DEVICE
            ),
            "truth": labels
        }
    with np.load(GRAPH, allow_pickle=False) as loaded:
        edges = torch.tensor(
            loaded["edge_index"],
            dtype=torch.long,
            device=DEVICE
        )
    assert edges.ndim == 2 and edges.shape[0] == 2
    assert edges.numel() > 0
    assert int(edges.min()) >= 0
    assert int(edges.max()) < len(drugs)
    counts = np.bincount(
        data["train"]["truth"],
        minlength=84
    ).astype(float)
    weights = np.clip(
        np.sqrt(counts.max() / (counts + 1)),
        0.5,
        3.0
    )
    weights /= weights.mean()
    np.testing.assert_allclose(
        weights,
        reference["class_weights"],
        rtol=0,
        atol=1e-12
    )
    loss_fn = nn.CrossEntropyLoss(
        weight=torch.tensor(
            weights,
            dtype=torch.float32,
            device=DEVICE
        )
    )
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    # Fresh initialization; no Original or Rewired weights loaded.
    model = CurrentNodeBNGCN().to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.0007,
        weight_decay=0.0001
    )
    assert sum(p.numel() for p in model.parameters()) == 1636244
    assert all(
        not module.track_running_stats
        for module in model.modules()
        if isinstance(module, nn.BatchNorm1d)
    )
    config = {
        "protocol": "current_node_bn_v2",
        "condition": "rewired",
        "seed": SEED,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "optimizer": "AdamW",
        "learning_rate": 0.0007,
        "weight_decay": 0.0001,
        "gradient_clip_norm": 5.0,
        "early_stopping": None,
        "scheduler": None,
        "architecture": str(model),
        "class_weights": weights.tolist(),
        "primary_selection": "calibration_macro_f1_all84",
        "tie_rule": "earliest epoch",
        "BN": "current full-node statistics in train and eval",
        "BN_track_running_stats": False,
        "BN_node_scope": "all 1705 fixed nodes; transductive",
        "descriptor_preprocessing_fit": "Train nodes only",
        "graph_source": "Train edges only",
        "shuffle_seed": "seed + 100000 * epoch",
        "feature_sha256": digest(cache),
        "graph_sha256": digest(GRAPH),
        "split_sha256": reference["split_sha256"],
        "versions": versions,
        "gpu": torch.cuda.get_device_name(0),
        "test_evaluated": False,
        "bitwise_GPU_resume_guaranteed": False
    }
    # All model and training settings must match corrected Original 42.
    for key in [
        "protocol", "epochs", "batch_size", "optimizer",
        "learning_rate", "weight_decay", "gradient_clip_norm",
        "early_stopping", "scheduler", "architecture", "class_weights",
        "primary_selection", "tie_rule", "BN", "BN_track_running_stats",
        "BN_node_scope", "descriptor_preprocessing_fit", "shuffle_seed"
    ]:
        assert config[key] == original_v2[key], f"Original mismatch: {key}"
    config_path = RUN / "config.json"
    if config_path.exists():
        saved_config = json.loads(config_path.read_text())
        differences = {
            key: [saved_config.get(key), config.get(key)]
            for key in set(saved_config) | set(config)
            if saved_config.get(key) != config.get(key)
        }
        assert set(differences) <= {"gpu"}, (
            f"Training configuration changed: {differences}"
        )
        # Preserve the exact original config bytes and checkpoint hash.
        config = saved_config
    else:
        save_json(config_path, config)
    # Append hardware provenance separately; never edit the original config.
    import platform
    from datetime import datetime, timezone
    provenance_path = RUN / "execution_segments.json"
    segments = json.loads(provenance_path.read_text()) if provenance_path.exists() else []
    segments.append({
        "utc": datetime.now(timezone.utc).isoformat(),
        "platform": "Kaggle",
        "python": platform.python_version(),
        "gpu": torch.cuda.get_device_name(0),
        "torch_cuda": torch.version.cuda,
        "versions": versions,
        "original_config_gpu": config["gpu"],
        "bitwise_continuation_guaranteed": False,
        "config_sha256": digest(config_path)
    })
    save_json(provenance_path, segments)
    config_hash = digest(config_path)
    done = RUN / "TRAINING_COMPLETE.json"
    if done.exists():
        completed = json.loads(done.read_text())
        assert completed["config_sha256"] == config_hash
        print("Run already completed.")
        print(json.dumps(completed, indent=2))
        print("\nSaved:", RUN)
        return
    last = RUN / "last_training_state.pt"
    epoch = 1
    next_batch = 0
    loss_sum = 0.0
    loss_batches = 0
    history = []
    best = {
        metric: -1.0
        for metric in ["macro_f1", "accuracy", "weighted_f1"]
    }
    best_epochs = {metric: None for metric in best}
    elapsed_before = 0.0
    if last.exists():
        saved = torch.load(
            last,
            map_location="cpu",
            weights_only=True
        )
        assert saved["config_sha256"] == config_hash
        model.load_state_dict(saved["model"])
        optimizer.load_state_dict(saved["optimizer"])
        epoch = saved["epoch"]
        next_batch = saved["next_batch"]
        loss_sum = saved["loss_sum"]
        loss_batches = saved["loss_batches"]
        history = saved["history"]
        best = saved["best"]
        best_epochs = saved["best_epochs"]
        elapsed_before = saved["elapsed_seconds"]
        torch.set_rng_state(saved["cpu_rng"])
        torch.cuda.set_rng_state(saved["cuda_rng"])
        del saved
        print(
            f"RESUME: Rewired 44 — epoch {epoch}, "
            f"completed batches {next_batch}"
        )
    else:
        print("START: Rewired 44 — corrected BN — FROM SCRATCH")
    started = time.perf_counter()
    def checkpoint():
        save_pt(last, {
            "config_sha256": config_hash,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "next_batch": next_batch,
            "loss_sum": loss_sum,
            "loss_batches": loss_batches,
            "history": history,
            "best": best,
            "best_epochs": best_epochs,
            "cpu_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state(),
            "elapsed_seconds": (
                elapsed_before + time.perf_counter() - started
            )
        })
    @torch.no_grad()
    def calibration_metrics():
        # Dropout off; BN uses current full-node statistics.
        model.eval()
        z = model.encode(x, edges)
        assert torch.isfinite(z).all()
        predictions = []
        pairs = data["calibration"]["pairs"]
        for start in range(0, len(pairs), 2048):
            logits = model.classify(z, pairs[start:start + 2048])
            assert torch.isfinite(logits).all()
            predictions.append(logits.argmax(1).cpu().numpy())
        pred = np.concatenate(predictions)
        truth = data["calibration"]["truth"]
        return {
            "accuracy": float(accuracy_score(truth, pred)),
            "macro_f1": float(f1_score(
                truth,
                pred,
                labels=np.arange(84),
                average="macro",
                zero_division=0
            )),
            "weighted_f1": float(f1_score(
                truth,
                pred,
                labels=np.arange(84),
                average="weighted",
                zero_division=0
            ))
        }
    n = len(data["train"]["pairs"])
    n_batches = math.ceil(n / BATCH_SIZE)
    progress = tqdm(
        total=EPOCHS,
        initial=min(epoch - 1, EPOCHS),
        desc="Rewired 44 — current-node BN"
    )
    while epoch <= EPOCHS:
        epoch_start = time.perf_counter()
        resumed = next_batch > 0
        generator = torch.Generator().manual_seed(
            SEED + 100000 * epoch
        )
        order = torch.randperm(
            n,
            generator=generator
        ).to(DEVICE)
        model.train()
        batches = tqdm(
            total=n_batches,
            initial=next_batch,
            desc=f"Epoch {epoch}/{EPOCHS}",
            leave=False
        )
        for batch in range(next_batch, n_batches):
            idx = order[
                batch * BATCH_SIZE:(batch + 1) * BATCH_SIZE
            ]
            optimizer.zero_grad(set_to_none=True)
            logits = model(
                x,
                edges,
                data["train"]["pairs"][idx]
            )
            loss = loss_fn(logits, data["train"]["y"][idx])
            assert torch.isfinite(loss), "Nonfinite loss."
            loss.backward()
            grad_norm = nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )
            assert torch.isfinite(grad_norm), "Nonfinite gradients."
            optimizer.step()
            loss_sum += float(loss.detach())
            loss_batches += 1
            next_batch = batch + 1
            batches.update(1)
            if next_batch % 10 == 0:
                batches.set_postfix(
                    loss=f"{loss_sum / loss_batches:.4f}"
                )
            if (
                next_batch % SAVE_EVERY == 0
                or next_batch == n_batches
            ):
                checkpoint()
        batches.close()
        metrics = calibration_metrics()
        for metric, value in metrics.items():
            if value > best[metric]:
                best[metric] = value
                best_epochs[metric] = epoch
                save_pt(RUN / f"best_by_{metric}.pt", {
                    "config_sha256": config_hash,
                    "model": model.state_dict(),
                    "epoch": epoch,
                    "criterion": metric,
                    "calibration_metrics": metrics,
                    "test_evaluated": False
                })
        row = {
            "epoch": epoch,
            "mean_batch_train_loss": loss_sum / loss_batches,
            "cal_accuracy": metrics["accuracy"],
            "cal_macro_f1": metrics["macro_f1"],
            "cal_weighted_f1": metrics["weighted_f1"],
            "epoch_segment_seconds": (
                time.perf_counter() - epoch_start
            ),
            "resumed_partway": resumed
        }
        history.append(row)
        epoch += 1
        next_batch = 0
        loss_sum = 0.0
        loss_batches = 0
        checkpoint()
        save_history(history)
        progress.update(1)
        progress.set_postfix(
            best_macro=f"{100 * best['macro_f1']:.2f}%"
        )
        tqdm.write(
            f"Epoch {row['epoch']:03d}/{EPOCHS} | "
            f"loss={row['mean_batch_train_loss']:.4f} | "
            f"Cal Acc={100 * metrics['accuracy']:.2f}% | "
            f"Macro-F1={100 * metrics['macro_f1']:.2f}% | "
            f"Weighted-F1={100 * metrics['weighted_f1']:.2f}% | saved"
        )
    progress.close()
    save_history(history)
    result = {
        "protocol": "current_node_bn_v2",
        "condition": "rewired",
        "seed": SEED,
        "completed_epochs": EPOCHS,
        "best_calibration_scores": best,
        "best_epochs": best_epochs,
        "primary_checkpoint": "best_by_macro_f1.pt",
        "elapsed_hours": (
            elapsed_before + time.perf_counter() - started
        ) / 3600,
        "test_evaluated": False,
        "config_sha256": config_hash
    }
    save_json(done, result)
    print("\n=== TRAINING COMPLETE ===")
    print(json.dumps(result, indent=2))
    print("\nSaved:", RUN)
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
main()


## Source cell 106

Original notebook zero-based index: 105.


In [ ]:
from pathlib import Path
import os
import json
import hashlib
import time
import math
import gc
import importlib.metadata as md
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
BASE = Path('/kaggle/input/datasets/amiramohammeddd/ddi-unique263-cal-test-stage2/DDI_KAGGLE_TRANSFER_SEED43/DDI_V2')
FEATURES = BASE / "PRIMARY_TOPOLOGY_TRAIN_ONLY_V2"
REFERENCE = FEATURES / "training_current_node_bn_v2/original/seed_42"
RUN = Path("/kaggle/working/DDI_V2/PRIMARY_TOPOLOGY_TRAIN_ONLY_V2/training_current_node_bn_v2/self_loops_only/seed_44")
def resolve_graph(filename, expected_hash=None):
    import hashlib
    import zipfile
    import os

    input_root = Path("/kaggle/input/datasets/amiramohammeddd/ddi-unique263-cal-test-stage2")
    unpacked = Path("/kaggle/working/SEED44_EXTRA_GRAPHS")
    unpacked.mkdir(parents=True, exist_ok=True)

    def sha(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for block in iter(lambda: f.read(1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    # Kaggle may keep the upload zipped or expose already-extracted files.
    # Read only matching graph members; never extract arbitrary archive paths.
    for archive_path in sorted(input_root.rglob("SEED44_EXTRA_GRAPHS*.zip")):
        with zipfile.ZipFile(archive_path, "r") as archive:
            for member in archive.infolist():
                if member.is_dir() or Path(member.filename).name != filename:
                    continue
                contents = archive.read(member)
                fingerprint = hashlib.sha256(contents).hexdigest()
                destination = unpacked / (fingerprint + "_" + filename)
                if destination.exists():
                    assert sha(destination) == fingerprint, "Extracted graph is damaged."
                else:
                    temporary = destination.with_suffix(".npz.tmp")
                    temporary.write_bytes(contents)
                    os.replace(temporary, destination)

    candidates = sorted(set(
        [p for p in Path("/kaggle/input").rglob(filename) if p.is_file()]
        + [p for p in unpacked.glob("*_" + filename) if p.is_file()]
    ))
    assert candidates, f"Missing graph: {filename}; attach SEED44_EXTRA_GRAPHS.zip."
    hashes = {p: sha(p) for p in candidates}
    if expected_hash is not None:
        candidates = [p for p in candidates if hashes[p] == expected_hash]
        assert candidates, f"No graph matches the recorded hash: {filename}"
    assert len({hashes[p] for p in candidates}) == 1, (
        f"Several different graphs named {filename}; report their paths: {candidates}"
    )
    return candidates[0]

GRAPH = resolve_graph("self_loops_only.npz")
SEED = 44
EPOCHS = 120
BATCH_SIZE = 512
SAVE_EVERY = 50
assert torch.cuda.is_available(), "A CUDA GPU is required."
assert BASE.exists(), "Input folder missing."
DEVICE = torch.device("cuda")
RUN.mkdir(parents=True, exist_ok=True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
def save_json(path, obj):
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False))
    os.replace(tmp, path)
def save_pt(path, obj):
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)
def save_history(history):
    tmp = RUN / "training_history.csv.tmp"
    pd.DataFrame(history).to_csv(tmp, index=False)
    os.replace(tmp, RUN / "training_history.csv")
class CurrentNodeBNGCN(nn.Module):
    def __init__(self):
        super().__init__()
        dims = [2250, 384, 384, 192]
        self.convs = nn.ModuleList([
            GCNConv(
                dims[i],
                dims[i + 1],
                improved=False,
                cached=False,
                add_self_loops=False,
                normalize=True,
                bias=True
            )
            for i in range(3)
        ])
        self.norms = nn.ModuleList([
            nn.BatchNorm1d(d, track_running_stats=False)
            for d in [384, 384, 192]
        ])
        self.head = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(256, 84)
        )
    def encode(self, x, edges):
        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            x = norm(conv(x, edges))
            if i < 2:
                x = F.dropout(
                    F.relu(x),
                    p=0.35,
                    training=self.training
                )
        return x
    def classify(self, z, pairs):
        a = z[pairs[:, 0]]
        b = z[pairs[:, 1]]
        pair_features = torch.cat(
            [a, b, (a - b).abs(), a * b],
            dim=1
        )
        return self.head(pair_features)
    def forward(self, x, edges, pairs):
        return self.classify(self.encode(x, edges), pairs)
def main():
    reference_path = REFERENCE / "config.json"
    cache = FEATURES / "train_only_feature_cache.npz"
    for path in [reference_path, cache, GRAPH]:
        assert path.exists(), f"Missing file: {path}"
    reference = json.loads(reference_path.read_text())
    assert reference["condition"] == "original"
    assert reference["seed"] == 42
    assert digest(cache) == reference["feature_sha256"]
    versions = {
        package: md.version(package)
        for package in [
            "torch",
            "torch-geometric",
            "numpy",
            "pandas",
            "scikit-learn"
        ]
    }
    original_v2_path = (
        FEATURES / "training_current_node_bn_v2/original/seed_42/config.json"
    )
    assert original_v2_path.exists(), "Corrected Original 42 config missing."
    original_v2 = json.loads(original_v2_path.read_text())
    assert original_v2["protocol"] == "current_node_bn_v2"
    assert original_v2["condition"] == "original"
    assert original_v2["seed"] == 42
    assert original_v2["feature_sha256"] == reference["feature_sha256"]
    assert original_v2["split_sha256"] == reference["split_sha256"]
    differences = {
        package: [original_v2["versions"].get(package), version]
        for package, version in versions.items()
        if original_v2["versions"].get(package) != version
    }
    assert not differences, (
        f"Environment differs from corrected Original 42: {differences}"
    )
    print("Environment matches corrected Original 42 ")
    with np.load(cache, allow_pickle=False) as loaded:
        x = torch.tensor(
            loaded["x"],
            dtype=torch.float32,
            device=DEVICE
        )
        drugs = loaded["all_drugs"].tolist()
    assert x.shape == (1705, 2250), f"Unexpected shape: {x.shape}"
    assert torch.isfinite(x).all()
    mapping = {drug: i for i, drug in enumerate(drugs)}
    assert len(mapping) == len(drugs), "Duplicate drug IDs in cache."
    data = {}
    # Only Train and Calibration are read. Test is not read.
    for split in ["train", "calibration"]:
        path = BASE / f"clean_{split}_split.csv"
        assert path.exists(), f"Missing file: {path}"
        assert digest(path) == reference["split_sha256"][split]
        frame = pd.read_csv(path)
        a = frame["Drug1"].map(mapping)
        b = frame["Drug2"].map(mapping)
        assert a.notna().all() and b.notna().all(), (
            f"Unmapped drug IDs in {split}."
        )
        labels = frame["label_id"].to_numpy(dtype=np.int64)
        assert set(labels) == set(range(84)), (
            f"Unexpected class coverage in {split}."
        )
        data[split] = {
            "pairs": torch.tensor(
                np.column_stack([a, b]).astype(np.int64),
                dtype=torch.long,
                device=DEVICE
            ),
            "y": torch.tensor(
                labels,
                dtype=torch.long,
                device=DEVICE
            ),
            "truth": labels
        }
    with np.load(GRAPH, allow_pickle=False) as loaded:
        edges = torch.tensor(
            loaded["edge_index"],
            dtype=torch.long,
            device=DEVICE
        )
    assert edges.ndim == 2 and edges.shape[0] == 2
    assert edges.numel() > 0
    assert int(edges.min()) >= 0
    assert int(edges.max()) < len(drugs)
    # Exactly one self-loop per node, with no inter-node edges.
    assert edges.shape[1] == len(drugs), "Expected one edge per node."
    assert torch.equal(edges[0], edges[1]), "Found non-self-loop edges."
    expected_nodes = torch.arange(len(drugs), device=DEVICE)
    assert torch.equal(torch.sort(edges[0]).values, expected_nodes), (
        "Self-loops must cover every node exactly once."
    )
    print(f"Graph verified: {len(drugs)} self-loops, no neighbor edges ")
    counts = np.bincount(
        data["train"]["truth"],
        minlength=84
    ).astype(float)
    weights = np.clip(
        np.sqrt(counts.max() / (counts + 1)),
        0.5,
        3.0
    )
    weights /= weights.mean()
    np.testing.assert_allclose(
        weights,
        reference["class_weights"],
        rtol=0,
        atol=1e-12
    )
    loss_fn = nn.CrossEntropyLoss(
        weight=torch.tensor(
            weights,
            dtype=torch.float32,
            device=DEVICE
        )
    )
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    # Fresh initialization; no Original or Rewired weights loaded.
    model = CurrentNodeBNGCN().to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.0007,
        weight_decay=0.0001
    )
    assert sum(p.numel() for p in model.parameters()) == 1636244
    assert all(
        not module.track_running_stats
        for module in model.modules()
        if isinstance(module, nn.BatchNorm1d)
    )
    config = {
        "protocol": "current_node_bn_v2",
        "condition": "self_loops_only",
        "seed": SEED,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "optimizer": "AdamW",
        "learning_rate": 0.0007,
        "weight_decay": 0.0001,
        "gradient_clip_norm": 5.0,
        "early_stopping": None,
        "scheduler": None,
        "architecture": str(model),
        "class_weights": weights.tolist(),
        "primary_selection": "calibration_macro_f1_all84",
        "tie_rule": "earliest epoch",
        "BN": "current full-node statistics in train and eval",
        "BN_track_running_stats": False,
        "BN_node_scope": "all 1705 fixed nodes; transductive",
        "descriptor_preprocessing_fit": "Train nodes only",
        "graph_source": "Self-loops only; no inter-node edges",
        "shuffle_seed": "seed + 100000 * epoch",
        "feature_sha256": digest(cache),
        "graph_sha256": digest(GRAPH),
        "split_sha256": reference["split_sha256"],
        "versions": versions,
        "gpu": torch.cuda.get_device_name(0),
        "test_evaluated": False,
        "bitwise_GPU_resume_guaranteed": False
    }
    # All model and training settings must match corrected Original 42.
    for key in [
        "protocol", "epochs", "batch_size", "optimizer",
        "learning_rate", "weight_decay", "gradient_clip_norm",
        "early_stopping", "scheduler", "architecture", "class_weights",
        "primary_selection", "tie_rule", "BN", "BN_track_running_stats",
        "BN_node_scope", "descriptor_preprocessing_fit", "shuffle_seed"
    ]:
        assert config[key] == original_v2[key], f"Original mismatch: {key}"
    config_path = RUN / "config.json"
    if config_path.exists():
        saved_config = json.loads(config_path.read_text())
        differences = {
            key: [saved_config.get(key), config.get(key)]
            for key in set(saved_config) | set(config)
            if saved_config.get(key) != config.get(key)
        }
        assert set(differences) <= {"gpu"}, (
            f"Training configuration changed: {differences}"
        )
        # Preserve the exact original config bytes and checkpoint hash.
        config = saved_config
    else:
        save_json(config_path, config)
    # Append hardware provenance separately; never edit the original config.
    import platform
    from datetime import datetime, timezone
    provenance_path = RUN / "execution_segments.json"
    segments = json.loads(provenance_path.read_text()) if provenance_path.exists() else []
    segments.append({
        "utc": datetime.now(timezone.utc).isoformat(),
        "platform": "Kaggle",
        "python": platform.python_version(),
        "gpu": torch.cuda.get_device_name(0),
        "torch_cuda": torch.version.cuda,
        "versions": versions,
        "original_config_gpu": config["gpu"],
        "bitwise_continuation_guaranteed": False,
        "config_sha256": digest(config_path)
    })
    save_json(provenance_path, segments)
    config_hash = digest(config_path)
    done = RUN / "TRAINING_COMPLETE.json"
    if done.exists():
        completed = json.loads(done.read_text())
        assert completed["config_sha256"] == config_hash
        print("Run already completed.")
        print(json.dumps(completed, indent=2))
        print("\nSaved:", RUN)
        return
    last = RUN / "last_training_state.pt"
    epoch = 1
    next_batch = 0
    loss_sum = 0.0
    loss_batches = 0
    history = []
    best = {
        metric: -1.0
        for metric in ["macro_f1", "accuracy", "weighted_f1"]
    }
    best_epochs = {metric: None for metric in best}
    elapsed_before = 0.0
    if last.exists():
        saved = torch.load(
            last,
            map_location="cpu",
            weights_only=True
        )
        assert saved["config_sha256"] == config_hash
        model.load_state_dict(saved["model"])
        optimizer.load_state_dict(saved["optimizer"])
        epoch = saved["epoch"]
        next_batch = saved["next_batch"]
        loss_sum = saved["loss_sum"]
        loss_batches = saved["loss_batches"]
        history = saved["history"]
        best = saved["best"]
        best_epochs = saved["best_epochs"]
        elapsed_before = saved["elapsed_seconds"]
        torch.set_rng_state(saved["cpu_rng"])
        torch.cuda.set_rng_state(saved["cuda_rng"])
        del saved
        print(
            f"RESUME: Self-loops only 44 — epoch {epoch}, "
            f"completed batches {next_batch}"
        )
    else:
        print("START: Self-loops only 44 — corrected BN — FROM SCRATCH")
    started = time.perf_counter()
    def checkpoint():
        save_pt(last, {
            "config_sha256": config_hash,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "next_batch": next_batch,
            "loss_sum": loss_sum,
            "loss_batches": loss_batches,
            "history": history,
            "best": best,
            "best_epochs": best_epochs,
            "cpu_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state(),
            "elapsed_seconds": (
                elapsed_before + time.perf_counter() - started
            )
        })
    @torch.no_grad()
    def calibration_metrics():
        # Dropout off; BN uses current full-node statistics.
        model.eval()
        z = model.encode(x, edges)
        assert torch.isfinite(z).all()
        predictions = []
        pairs = data["calibration"]["pairs"]
        for start in range(0, len(pairs), 2048):
            logits = model.classify(z, pairs[start:start + 2048])
            assert torch.isfinite(logits).all()
            predictions.append(logits.argmax(1).cpu().numpy())
        pred = np.concatenate(predictions)
        truth = data["calibration"]["truth"]
        return {
            "accuracy": float(accuracy_score(truth, pred)),
            "macro_f1": float(f1_score(
                truth,
                pred,
                labels=np.arange(84),
                average="macro",
                zero_division=0
            )),
            "weighted_f1": float(f1_score(
                truth,
                pred,
                labels=np.arange(84),
                average="weighted",
                zero_division=0
            ))
        }
    n = len(data["train"]["pairs"])
    n_batches = math.ceil(n / BATCH_SIZE)
    progress = tqdm(
        total=EPOCHS,
        initial=min(epoch - 1, EPOCHS),
        desc="Self-loops only 44 — current-node BN"
    )
    while epoch <= EPOCHS:
        epoch_start = time.perf_counter()
        resumed = next_batch > 0
        generator = torch.Generator().manual_seed(
            SEED + 100000 * epoch
        )
        order = torch.randperm(
            n,
            generator=generator
        ).to(DEVICE)
        model.train()
        batches = tqdm(
            total=n_batches,
            initial=next_batch,
            desc=f"Epoch {epoch}/{EPOCHS}",
            leave=False
        )
        for batch in range(next_batch, n_batches):
            idx = order[
                batch * BATCH_SIZE:(batch + 1) * BATCH_SIZE
            ]
            optimizer.zero_grad(set_to_none=True)
            logits = model(
                x,
                edges,
                data["train"]["pairs"][idx]
            )
            loss = loss_fn(logits, data["train"]["y"][idx])
            assert torch.isfinite(loss), "Nonfinite loss."
            loss.backward()
            grad_norm = nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )
            assert torch.isfinite(grad_norm), "Nonfinite gradients."
            optimizer.step()
            loss_sum += float(loss.detach())
            loss_batches += 1
            next_batch = batch + 1
            batches.update(1)
            if next_batch % 10 == 0:
                batches.set_postfix(
                    loss=f"{loss_sum / loss_batches:.4f}"
                )
            if (
                next_batch % SAVE_EVERY == 0
                or next_batch == n_batches
            ):
                checkpoint()
        batches.close()
        metrics = calibration_metrics()
        for metric, value in metrics.items():
            if value > best[metric]:
                best[metric] = value
                best_epochs[metric] = epoch
                save_pt(RUN / f"best_by_{metric}.pt", {
                    "config_sha256": config_hash,
                    "model": model.state_dict(),
                    "epoch": epoch,
                    "criterion": metric,
                    "calibration_metrics": metrics,
                    "test_evaluated": False
                })
        row = {
            "epoch": epoch,
            "mean_batch_train_loss": loss_sum / loss_batches,
            "cal_accuracy": metrics["accuracy"],
            "cal_macro_f1": metrics["macro_f1"],
            "cal_weighted_f1": metrics["weighted_f1"],
            "epoch_segment_seconds": (
                time.perf_counter() - epoch_start
            ),
            "resumed_partway": resumed
        }
        history.append(row)
        epoch += 1
        next_batch = 0
        loss_sum = 0.0
        loss_batches = 0
        checkpoint()
        save_history(history)
        progress.update(1)
        progress.set_postfix(
            best_macro=f"{100 * best['macro_f1']:.2f}%"
        )
        tqdm.write(
            f"Epoch {row['epoch']:03d}/{EPOCHS} | "
            f"loss={row['mean_batch_train_loss']:.4f} | "
            f"Cal Acc={100 * metrics['accuracy']:.2f}% | "
            f"Macro-F1={100 * metrics['macro_f1']:.2f}% | "
            f"Weighted-F1={100 * metrics['weighted_f1']:.2f}% | saved"
        )
    progress.close()
    save_history(history)
    result = {
        "protocol": "current_node_bn_v2",
        "condition": "self_loops_only",
        "seed": SEED,
        "completed_epochs": EPOCHS,
        "best_calibration_scores": best,
        "best_epochs": best_epochs,
        "primary_checkpoint": "best_by_macro_f1.pt",
        "elapsed_hours": (
            elapsed_before + time.perf_counter() - started
        ) / 3600,
        "test_evaluated": False,
        "config_sha256": config_hash
    }
    save_json(done, result)
    print("\n=== TRAINING COMPLETE ===")
    print(json.dumps(result, indent=2))
    print("\nSaved:", RUN)
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
main()


## Source cell 107

Original notebook zero-based index: 106.


In [ ]:
# CELL 5 — Verify completion and export all three result folders.
from pathlib import Path
import json
import os
import zipfile
from IPython.display import FileLink, display

BASE = Path("/kaggle/working/DDI_V2")
TRAINING = BASE / "PRIMARY_TOPOLOGY_TRAIN_ONLY_V2/training_current_node_bn_v2"
conditions = ["original", "rewired", "self_loops_only"]
selected = []
for condition in conditions:
    run = TRAINING / condition / "seed_44"
    done = run / "TRAINING_COMPLETE.json"
    assert done.is_file(), f"Not complete yet: {condition}"
    result = json.loads(done.read_text())
    assert result["condition"] == condition and result["seed"] == 44
    assert result["completed_epochs"] == 120 and result["test_evaluated"] is False
    print(condition, "Calibration Macro-F1:", 100 * result["best_calibration_scores"]["macro_f1"])
    selected.extend(p for p in sorted(run.rglob("*")) if p.is_file() and not p.name.endswith(".tmp"))

out = Path("/kaggle/working/KAGGLE_SEED44_ALL3_COMPLETE.zip")
tmp = out.with_suffix(".zip.tmp")
with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=3) as archive:
    for path in selected:
        archive.write(path, arcname=(Path("DDI_V2") / path.relative_to(BASE)).as_posix())
with zipfile.ZipFile(tmp, "r") as archive:
    assert archive.testzip() is None
os.replace(tmp, out)
print(f"Backup ready: {out.stat().st_size / 1024**2:.2f} MB")
os.chdir("/kaggle/working")
display(FileLink(out.name))
